# Transformers 模型量化技术：AWQ

![img](https://huggingface.co/datasets/ybelkada/documentation-images/resolve/main/Thumbnail.png)

在2023年6月，Ji Lin等人发表了论文[AWQ：Activation-aware Weight Quantization for LLM Compression and Acceleration](https://arxiv.org/pdf/2306.00978.pdf)。

这篇论文详细介绍了一种激活感知权重量化算法，可以用于压缩任何基于 Transformer 的语言模型，同时只有微小的性能下降。关于 AWQ 算法的详细介绍，见[MIT Han Song 教授分享](https://hanlab.mit.edu/projects/awq)。

transformers 现在支持两个不同的 AWQ 开源实现库：

- [AutoAWQ](https://github.com/casper-hansen/AutoAWQ)
- [LLM-AWQ](https://github.com/mit-han-lab/llm-awq) 


因为 LLM-AWQ 不支持 Nvidia T4 GPU（课程演示 GPU），所以我们使用 AutoAWQ 库来介绍和演示 AWQ 模型量化技术。

## 量化前模型测试文本生成任务

In [1]:
from transformers import pipeline

model_path = "facebook/opt-125m"

# 使用 GPU 加载原始的 OPT-125m 模型
generator = pipeline('text-generation',
                     model=model_path,
                     device=0,
                     do_sample=True,
                     num_return_sequences=3)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


#### 实测GPU显存占用：加载 OPT-125m 模型后

```shell
Sun Dec 24 15:11:33 2023
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:0D.0 Off |                    0 |
| N/A   47C    P0              26W /  70W |    635MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+

+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|
```

In [2]:
generator("The woman worked as a")

[{'generated_text': 'The woman worked as a bus operator for several years during her 20s before going back to school.'},
 {'generated_text': "The woman worked as a dishwasher. I worked in a restaurant.   She didn't ask"},
 {'generated_text': 'The woman worked as a janitor, so she can also do this for a living.\nI'}]

In [3]:
generator("The man worked as a")

[{'generated_text': 'The man worked as a manager for an apartment building in Brooklyn. He is described as having an aggressive'},
 {'generated_text': 'The man worked as a bartender and now does bartending in New Hampshire. What he is doing is'},
 {'generated_text': "The man worked as a lawyer. I'd be okay with him in a role that he's never"}]

## 使用 AutoAWQ 量化模型

下面我们以 `facebook opt-125m` 模型为例，使用 `AutoAWQ` 库实现的 AWQ 算法实现模型量化。

In [4]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer


quant_path = "models/opt-125m-awq"
# "w_bit": 4 将模型的权重从 FP16（16-bit）压缩到 4-bit 整数
# "version": "GEMM" 计算内核版本，这指定了推理时使用哪种底层 CUDA 内核来执行 4-bit 的矩阵运算
# "zero_point": True 是否使用零点
### True（非对称量化）：除了缩放因子，还为每个组引入一个零点偏移量。这能更好地处理分布不对称的权重（比如大部分为正、少数为负的权重）。对于大多数 LLM，权重分布并非严格对称，所以开启 zero_point 通常能带来更好的精度。
### False（对称量化）：不使用零点，量化范围关于 0 对称。这通常用于某些特定内核（如 Marlin），因为 Marlin 内核在对称量化下能获得最高性能。
### 注意：使用 GEMM 内核时，zero_point 通常设为 True；使用 Marlin 内核时，则必须设为 False。两者不兼容。
quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}

# 加载模型
model = AutoAWQForCausalLM.from_pretrained(model_path, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [5]:
# 量化模型
model.quantize(tokenizer, quant_config=quant_config)

Repo card metadata block was not found. Setting CardData to empty.


Generating validation split: 0 examples [00:00, ? examples/s]

AWQ: 100%|██████████| 12/12 [00:40<00:00,  3.34s/it]


#### 实测GPU显存使用：量化模型时峰值达到将近 4GB

```shell
Sun Dec 24 15:12:50 2023
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:0D.0 Off |                    0 |
| N/A   48C    P0              32W /  70W |    3703MiB / 15360MiB |      2%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+

+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|
```

In [6]:
quant_config

{'zero_point': True, 'q_group_size': 128, 'w_bit': 4, 'version': 'GEMM'}

#### Transformers 兼容性配置

为了使`quant_config` 与 transformers 兼容，我们需要修改配置文件：`使用 Transformers.AwqConfig 来实例化量化模型配置`

“不兼容”的并不是数据本身，而是 autoawq 和 transformers 这两个库对“量化配置”的“方言”不同。这段代码就是一个“翻译”步骤，它的核心作用，是让一个由 autoawq 量化的模型，在加载时能被 transformers 框架正确识别和反量化。

1. autoawq：负责“生产”：transformers 官方文档明确指出，它不能直接用来量化模型，量化工作必须依赖 AutoAWQ 这个专用库来完成。autoawq 在量化后，会生成自己的一套配置文件。
2. transformers：负责“消费”：transformers 的 from_pretrained 方法支持加载超过20种量化模型，但它只认自己定义的标准配置格式。当你加载一个已经量化好的检查点时，transformers 会去读取模型 config.json 里的 quantization_config 字段，以确定该如何处理这些4-bit权重。

In [7]:
from transformers import AwqConfig, AutoConfig

# 修改配置文件以使其与transformers集成兼容
quantization_config = AwqConfig(
    bits=quant_config["w_bit"],
    group_size=quant_config["q_group_size"],
    zero_point=quant_config["zero_point"],
    version=quant_config["version"].lower(),
).to_dict()

# 预训练的transformers模型存储在model属性中，我们需要传递一个字典
model.model.config.quantization_config = quantization_config

In [8]:
# 保存模型权重
model.save_quantized(quant_path)
tokenizer.save_pretrained(quant_path)  # 保存分词器

('models/opt-125m-awq/tokenizer_config.json',
 'models/opt-125m-awq/special_tokens_map.json',
 'models/opt-125m-awq/vocab.json',
 'models/opt-125m-awq/merges.txt',
 'models/opt-125m-awq/added_tokens.json',
 'models/opt-125m-awq/tokenizer.json')

### 使用 GPU 加载量化模型

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(quant_path)
model = AutoModelForCausalLM.from_pretrained(quant_path, device_map="cuda").to(0)

In [10]:
def generate_text(text):
    inputs = tokenizer(text, return_tensors="pt").to(0)

    out = model.generate(**inputs, max_new_tokens=64)
    return tokenizer.decode(out[0], skip_special_tokens=True)


In [11]:
result = generate_text("Merry Christmas! I'm glad to")
print(result)

Merry Christmas! I'm glad to see you on this one. (C is always on your a a nice. (C's is always on. (R is always on  is! and   is and  is on,   on is on  on   is  on  is on .  is do  on  is


In [12]:
result = generate_text("The woman worked as a")
print(result)

The woman worked as a a and is still working against one?


*
, is and is


.

.

- was

, - is

.

.

-

x/ *

1

.

.

4

.




## Homework：使用 AWQ 算法量化 Facebook OPT-2.7B 模型

Facebook OPT 模型：https://huggingface.co/facebook?search_models=opt

In [1]:
import gc
import os
import time
from pathlib import Path

os.environ.setdefault(
    "PYTORCH_CUDA_ALLOC_CONF",
    "expandable_segments:True",
)

import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from gptqmodel import GPTQModel, BACKEND
from gptqmodel.quantization import AWQConfig, FORMAT
from gptqmodel.nn_modules.qlinear.torch_awq import (
    AwqTorchLinear,
)

/home/sigan/miniconda3/envs/nowLLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


W1104 23:02:04.576000 66683 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1104 23:02:04.694000 66683 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.7.0
Transformers : 5.20.2
Torch        : 2.14.1+cu130
Triton       : 3.8.0


In [2]:
model_id = "facebook/opt-2.7b"

quant_path = Path(
    "/sigan/ai-ml/models/quantized/"
    "opt-2.7b-awq-4bit"
)

quant_path.mkdir(
    parents=True,
    exist_ok=True,
)

print("模型：", model_id)
print("输出：", quant_path)

模型： facebook/opt-2.7b
输出： /sigan/ai-ml/models/quantized/opt-2.7b-awq-4bit


### 校准数据

In [3]:
wiki_dataset = load_dataset(
    "wikitext",
    "wikitext-2-raw-v1",
    split="train",
)

calibration_dataset = [
    text.strip()
    for text in wiki_dataset["text"]
    if len(text.strip()) > 100
][:128]

print("校准样本数：", len(calibration_dataset))
print(
    "第一条样本：",
    calibration_dataset[0][:300],
)

INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/Salesforce/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/api/resolve-cache/datasets/Salesforce/wikitext/b08601e04326c79dfdd32d625aee71d232d685c3/README.md?%2Fdatasets%2FSalesforce%2Fwikitext%2Fresolve%2Fmain%2FREADME.md=&etag=%222a4fec2bc8df76c9d4da1c8e8865b625eb221c76%22 "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 307 Temporary Redirect"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/Salesforce/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 404 Not Found"
Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub
Found th

校准样本数： 128
第一条样本： Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in Janua


### 创建AWQ配置

| 参数 | 作用 |
|---|---|
| `bits=4` | 权重压成 4-bit |
| `group_size=128` | 每 128 个输入通道共享量化尺度 |
| `sym=False` | 使用非对称量化，等价于 AutoAWQ 的 `zero_point=True` |
| `format=FORMAT.GEMM` | 输出 AWQ GEMM 格式，兼容性较广 |
| `scale_search_chunked_activations=True` | 分块处理激活搜索，降低峰值内存 |
| `offload_to_disk=False` | 2.7B 可以放得下，不需要磁盘 offload |

In [4]:
quant_config = AWQConfig(
    bits=4,
    group_size=128,
    sym=False,
    format=FORMAT.GEMM,
    scale_search_chunked_activations=True,
    offload_to_disk=False,
    device="cpu",
)

### 加载原模型并量化

In [5]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"

In [6]:
torch.cuda.reset_peak_memory_stats()

model = GPTQModel.load(
    model_id,
    quantize_config=quant_config,
    dtype=torch.float16,
    backend=BACKEND.AWQ_TORCH,
)

print("model_type：", model.model.config.model_type)

INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/revision/main "HTTP/1.1 200 OK"
Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 1298.18it/s]
INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]
INFO  Loader: loading model directly to CPU (not using meta device or turtle_model)


Loading weights: 100%|██████████| 517/517 [00:00<00:00, 11920.94it/s]
INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "eos_token_id": 2,
  "pad_token_id": 1
}

INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "do_sample": true,
  "eos_token_id": 2,
  "pad_token_id": 1
}

INFO  Kernel: loaded -> `[]`                                                    
model_type： opt


### 开始量化

In [7]:
print(model.qlinear_kernel)

None


In [8]:
model.qlinear_kernel = AwqTorchLinear

print(model.qlinear_kernel)

<class 'gptqmodel.nn_modules.qlinear.torch_awq.AwqTorchLinear'>


In [9]:
import time
import torch

start = time.perf_counter()

model.quantize(
    calibration_dataset,
    batch_size=1,
)

elapsed = time.perf_counter() - start

print(f"量化耗时：{elapsed / 60:.2f} 分钟")

INFO  Packing Kernel: selected: `AwqTorchLinear`                                
INFO  Packing Kernel: selected: `AwqTorchLinear`                                
WARN  Calibration dataset size should be more than 256. Current: 128.           
INFO  Calibration: Sort in descending order by length                           
INFO  Calibration: Total padded tokens: 0                                       
INFO  Calibration: Total non-padded tokens: 19406                               
INFO  Calibration: Total tokens: 19406                                          
WARN  The average length of input_ids of calibration_dataset should be greater than 256: actual avg: 151.609375.
INFO  ModuleLooper: capturing layer inputs from 128 calibration batches         
INFO  +------------+-------+--------+-------+---------+--------+---------+      
INFO  | region     | count | last_s | avg_s | total_s | pct    | source  |      
INFO  +------------+-------+--------+-------+---------+--------+---------+   

AWQProcessor: layer 0 fallback quant 6 modules: ['fc2', 'self_attn.q_proj', 'fc1', 'self_attn.out_proj', 'self_attn.k_proj', 'self_attn.v_proj']


INFO  +-+--+--+--+---+-+-+-+-+-+--+-+-----+------+---+----+--+----+-----+--+---+
INFO  |p|la|mo|fe|...|l|s|d|t|f|(v|d|ac...|act...|...|a...|sc|s...|sc...|sc|...|
INFO  +-+--+--+--+---+-+-+-+-+-+--+-+-----+------+---+----+--+----+-----+--+---+
INFO  |aw|0|fc|102...|f16...|n|...| |0| |cu| |...|22|12|la...|f|...|22 |1|la...|
INFO  +--+-+--+------+------+-+---+-+-+-+--+-+---+--+--+-----+-+---+---+-+-----+
INFO  |aw|0|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |a|0|f|25...|f1...|...|19| |...| |...| |19|22|1|lat...|fc1|19|22|1|lat...|
INFO  +-+-+-+-----+-----+---+--+-+---+-+---+-+--+--+-+------+---+--+--+-+------+
INFO  |aw|0|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|0|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-

AWQProcessor: layer 1 fallback quant 6 modules: ['fc2', 'self_attn.q_proj', 'fc1', 'self_attn.out_proj', 'self_attn.k_proj', 'self_attn.v_proj']


INFO  |p|la|mo|fe|...|l|s|d|t|f|(v|d|ac...|act...|...|a...|sc|s...|sc...|sc|...|
INFO  +-+--+--+--+---+-+-+-+-+-+--+-+-----+------+---+----+--+----+-----+--+---+
INFO  |aw|1|fc|102...|f16...|n|...| |0| |cu| |...|22|12|la...|f|...|22 |1|la...|
INFO  +--+-+--+------+------+-+---+-+-+-+--+-+---+--+--+-----+-+---+---+-+-----+
INFO  |aw|1|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |a|1|f|25...|f1...|...|19| |...| |...| |19|22|1|lat...|fc1|19|22|1|lat...|
INFO  +-+-+-+-----+-----+---+--+-+---+-+---+-+--+--+-+------+---+--+--+-+------+
INFO  |aw|1|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|1|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|1|se|2...|f16...|n

AWQProcessor: layer 2 fallback quant 6 modules: ['fc2', 'self_attn.q_proj', 'fc1', 'self_attn.out_proj', 'self_attn.k_proj', 'self_attn.v_proj']


... (55 outputs left out)

AWQProcessor: layer 30 fallback quant 6 modules: ['fc2', 'self_attn.q_proj', 'fc1', 'self_attn.out_proj', 'self_attn.k_proj', 'self_attn.v_proj']


INFO  |p|la|mo|fe|...|l|s|d|t|f|(v|d|ac...|act...|...|a...|sc|s...|sc...|sc|...|
INFO  +-+--+--+--+---+-+-+-+-+-+--+-+-----+------+---+----+--+----+-----+--+---+
INFO  |aw|3|fc|102...|f16...|n|...| |0| |cu| |...|22|12|la...|f|...|22 |1|la...|
INFO  +--+-+--+------+------+-+---+-+-+-+--+-+---+--+--+-----+-+---+---+-+-----+
INFO  |aw|3|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |a|30|f|25...|f1...|...|19| |0.| |...| |19|22|1|lat...|fc1|19|22|1|lat...|
INFO  +-+--+-+-----+-----+---+--+-+--+-+---+-+--+--+-+------+---+--+--+-+------+
INFO  |aw|3|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|3|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|3|se|2...|f16...|n

AWQProcessor: layer 31 fallback quant 6 modules: ['fc2', 'self_attn.q_proj', 'fc1', 'self_attn.out_proj', 'self_attn.k_proj', 'self_attn.v_proj']


INFO  |p|la|mo|fe|...|l|s|d|t|f|(v|d|ac...|act...|...|a...|sc|s...|sc...|sc|...|
INFO  +-+--+--+--+---+-+-+-+-+-+--+-+-----+------+---+----+--+----+-----+--+---+
INFO  |aw|3|fc|102...|f16...|n|...| |0| |cu| |...|22|12|la...|f|...|22 |1|la...|
INFO  +--+-+--+------+------+-+---+-+-+-+--+-+---+--+--+-----+-+---+---+-+-----+
INFO  |aw|3|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |a|31|f|25...|f1...|...|19| |0.| |...| |19|22|1|lat...|fc1|19|22|1|lat...|
INFO  +-+--+-+-----+-----+---+--+-+--+-+---+-+--+--+-+------+---+--+--+-+------+
INFO  |aw|3|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|3|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|3|se|2...|f16...|n

### 保存模型

In [10]:
model.save(str(quant_path))

tokenizer = AutoTokenizer.from_pretrained(
    model_id
)

tokenizer.save_pretrained(
    str(quant_path)
)

print("AWQ 模型已保存：", quant_path)

Writing model shards: 0it [00:00, ?it/s]

INFO  Saved Quantize Config: 
{
  "bits": 4,
  "group_size": 128,
  "desc_act": false,
  "lm_head": false,
  "method": "awq",
  "quant_method": "awq",
  "format": "gemm",
  "checkpoint_format": "gemm",
  "pack_dtype": "int32",
  "meta": {
    "quantizer": [
      "gptqmodel:7.7.0"
    ],
    "uri": "https://github.com/modelcloud/gptqmodel",
    "damp_percent": null,
    "damp_auto_increment": null,
    "static_groups": null,
    "true_sequential": true,
    "mse": null,
    "gptaq": null,
    "foem": null,
    "act_group_aware": null,
    "fallback": {
      "strategy": "rtn",
      "threshold": "0.5%",
      "smooth": null
    },
    "offload_to_disk": false,
    "telemetry": {
      "device": false
    },
    "offload_to_disk_path": null,
    "pack_impl": "auto",
    "gc_mode": "interval",
    "wait_for_submodule_finalizers": false,
    "auto_forward_data_parallel": true,
    "dense_vram_strategy": "exclusive",
    "dense_vram_strategy_devices": null,
    "moe_vram_strategy": "exclus

Files in directory:
quant_log.csv
config.json
generation_config.json
quantize_config.json
Content of saved `generation_config.json`:
{
    "_from_model_config": true,
    "bos_token_id": 2,
    "do_sample": true,
    "eos_token_id": 2,
    "pad_token_id": 1,
    "transformers_version": "5.20.2"
}
Content of saved `config.json`:
{
    "_remove_final_layer_norm": false,
    "activation_dropout": 0.0,
    "activation_function": "relu",
    "architectures": [
        "OPTForCausalLM"
    ],
    "attention_dropout": 0.0,
    "bos_token_id": 2,
    "do_layer_norm_before": true,
    "dropout": 0.1,
    "dtype": "float16",
    "enable_bias": true,
    "eos_token_id": 2,
    "ffn_dim": 10240,
    "hidden_size": 2560,
    "init_std": 0.02,
    "layer_norm_elementwise_affine": true,
    "layerdrop": 0.0,
    "max_position_embeddings": 2048,
    "model_type": "opt",
    "num_attention_heads": 32,
    "num_hidden_layers": 32,
    "pad_token_id": 1,
    "prefix": "</s>",
    "quantization_config": {

INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


AWQ 模型已保存： /sigan/ai-ml/models/quantized/opt-2.7b-awq-4bit


### 重新加载 AWQ 模型

In [11]:
from gptqmodel import BACKEND, GPTQModel

tokenizer = AutoTokenizer.from_pretrained(
    str(quant_path),
    local_files_only=True,
)

reloaded_model = GPTQModel.load(
    str(quant_path),
    device="cuda:0",
    dtype=torch.float16,
    backend=BACKEND.AWQ_TORCH,
)

hf_model = reloaded_model.model
hf_model.eval()

print("推理后端：", BACKEND.AWQ_TORCH)

from_quantized: adapter: None
INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  The layer model.decoder.embed_tokens is not quantized.                    
INFO  The layer model.decoder.embed_positions is not quantized.                 
INFO  Kernel: selected: `AwqTorchLinear`                                        
INFO  Kernel: candidates -> `[AwqTorchLinear]`                                  
INFO  Kernel: per-module selections -> `[AwqTorchLinear=192]`                   
INFO  Kernel: selected -> `AwqTorchLinear`.                                     
INFO  Loader: device = cuda:0                                                   
INFO  Loader: honoring explicit device_map request: {'': 'cuda:0'}              
INFO  Loader: device_map = {'': 'cuda:0'}                                       


INFO  Kernel: selected: `AwqTorchLinear`                                        
INFO  gc.collect() reclaimed 10 objects in 0.133s                               


INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 1,
  "use_cache": true
}

INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "do_sample": true,
  "eos_token_id": 2,
  "pad_token_id": 1
}

INFO  Kernel: loaded -> `[AwqTorchLinear]`                                      
推理后端： BACKEND.AWQ_TORCH


In [12]:
prompt = "Merry Christmas! I'm glad to"

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to("cuda:0")

with torch.no_grad():
    output_ids = hf_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print(
    tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
    )
)

Merry Christmas! I'm glad to see you're still around.
I'm still here, just not posting as much. I'm still here, just not posting as much. I'm still here, just not posting as much.
I'm glad to hear that. I hope you're doing well.


### 检查量化层

In [14]:
q_proj = (
    hf_model.model
    .decoder.layers[0]
    .self_attn.q_proj
)

print("类型：", type(q_proj).__name__)
print("输入维度：", q_proj.in_features)
print("输出维度：", q_proj.out_features)

print(
    "qweight：",
    q_proj.qweight.shape,
    q_proj.qweight.dtype,
)

print(
    "qzeros：",
    q_proj.qzeros.shape,
    q_proj.qzeros.dtype,
)

print(
    "scales：",
    q_proj.scales.shape,
    q_proj.scales.dtype,
)

类型： AwqTorchLinear
输入维度： 2560
输出维度： 2560
qweight： torch.Size([2560, 320]) torch.int32
qzeros： torch.Size([20, 320]) torch.int32
scales： torch.Size([20, 2560]) torch.float16


### 查看模型大小

In [15]:
def directory_size_gb(path):
    return sum(
        file.stat().st_size
        for file in Path(path).rglob("*")
        if file.is_file()
    ) / 1024**3


print(
    "AWQ 模型目录大小 GB：",
    directory_size_gb(quant_path),
)

print(
    "模型张量近似占用 GB：",
    hf_model.get_memory_footprint() / 1024**3,
)

AWQ 模型目录大小 GB： 1.47255914285779
模型张量近似占用 GB： 1.4691352844238281
